In [1]:
!pip install tf2onnx
import zipfile
import os
import subprocess

zip_path = "/content/converted_savedmodel.zip"
extracted_dir = "/content/extracted_savedmodel"
onnx_output_path = "model.onnx"

# 1. Clear and extract zip archive
if os.path.exists(extracted_dir):
    import shutil
    shutil.rmtree(extracted_dir)

print("Extracting SavedModel zip file...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

# 2. Automatically locate the folder containing 'saved_model.pb'
actual_model_dir = None
for root, dirs, files in os.walk(extracted_dir):
    if "saved_model.pb" in files:
        actual_model_dir = root
        break

if not actual_model_dir:
    raise FileNotFoundError("Could not find 'saved_model.pb' inside the extracted zip. Check your zip contents.")

print(f"Found SavedModel at: {actual_model_dir}")

# 3. Convert to ONNX using subprocess with the correct path
print("Converting TensorFlow SavedModel to ONNX...")
cmd = [
    "python", "-m", "tf2onnx.convert",
    "--saved-model", actual_model_dir,
    "--output", onnx_output_path,
    "--opset", "13"
]

subprocess.run(cmd, check=True)
print(f"Model successfully converted and saved to: {onnx_output_path}")

# 4. Generate your labels file
labels_output_path = "labels.txt"
class_labels = [
    "0 obstacle",
    "1 path"
]

with open(labels_output_path, "w") as f:
    for label in class_labels:
        f.write(f"{label}\n")
print(f"Generated '{labels_output_path}' successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 81.6 MB/s eta 0:00:00
Extracting SavedModel zip file...
Found SavedModel at: /content/extracted_savedmodel/model.savedmodel
Converting TensorFlow SavedModel to ONNX...
Model successfully converted and saved to: model.onnx
Generated 'labels.txt' successfully.
